In [1]:
import pandas as pd
import numpy as np

from utils.config import load_config
from preprocessing.house_prices_preprocessing import HousePricesPreprocessor
from models.advanced_tuning import run_optuna_catboost

In [2]:
config = load_config()

data = pd.read_csv(config["paths"]["train"])

# --- разделяем target ---
target = "SalePrice"

# логарифмируем таргет
y = np.log1p(data[target])

X = data.drop(columns=[target])

# --- preprocessing ---
preprocessor = HousePricesPreprocessor(outlier_quantile=0.95)

X_prepared = preprocessor.fit_transform(X)

X_prepared.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageFinish_Unf,GarageFinish_Fin,GarageFinish_None,BsmtFinType1_GLQ,BsmtFinType1_ALQ,BsmtFinType1_Unf,BsmtFinType1_Rec,BsmtFinType1_BLQ,BsmtFinType1_None,BsmtFinType1_LwQ
0,0.073375,-0.208804,-0.332210,0.651479,-0.517200,1.050994,0.878668,0.739648,0.667140,-0.327561,...,0,0,0,1,0,0,0,0,0,0
1,-0.872563,0.646406,-0.009592,-0.071836,2.179628,0.156734,-0.429577,-0.654947,1.327216,-0.327561,...,0,0,0,0,1,0,0,0,0,0
2,0.073375,-0.037762,0.453294,0.651479,-0.517200,0.984752,0.830215,0.497729,0.133255,-0.327561,...,0,0,0,1,0,0,0,0,0,0
3,0.309859,-0.493874,-0.023619,0.651479,-0.517200,-1.863632,-0.720298,-0.654947,-0.521967,-0.327561,...,1,0,0,0,1,0,0,0,0,0
4,0.073375,0.874462,1.297710,1.374795,-0.517200,0.951632,0.733308,1.835402,0.543376,-0.327561,...,0,0,0,1,0,0,0,0,0,0


In [3]:
study, results_df = run_optuna_catboost(
    X=X_prepared,
    y=y,
    n_trials=500,
    n_splits=5,
)

Optuna tuning:   0%|          | 0/500 [00:00<?, ?it/s]

In [4]:
print("Лучшие параметры:", study.best_params)
print("Лучший RMSE:", study.best_value)

Лучшие параметры: {'iterations': 920, 'learning_rate': 0.06152055348604786, 'depth': 4, 'l2_leaf_reg': 7, 'random_strength': 3, 'subsample': 0.8}
Лучший RMSE: 0.12100098503803247


In [5]:
results_df.head()

,rank,number,mean_rmse,mean_best_rmse,mean_best_iteration,best_iteration_list,state,iterations,learning_rate,depth,l2_leaf_reg,random_strength,subsample
0,1,166,0.121001,0.121001,637.8,"[782, 847, 159, 732, 669]",TrialState.COMPLETE,920,0.061521,4,7,3,0.8
1,2,313,0.121124,0.121124,679.0,"[791, 980, 143, 809, 672]",TrialState.COMPLETE,980,0.060457,4,7,3,0.8
2,3,409,0.121125,0.121125,670.2,"[798, 847, 163, 821, 722]",TrialState.COMPLETE,960,0.057911,4,7,3,0.8
3,4,220,0.121135,0.121135,666.6,"[791, 847, 160, 596, 939]",TrialState.COMPLETE,940,0.058691,4,7,3,0.8
4,5,231,0.121183,0.121183,637.2,"[783, 870, 165, 668, 700]",TrialState.COMPLETE,900,0.058279,4,7,3,0.8


# Результаты подбора гиперпараметров CatBoost (Optuna)

## Общая информация

В данном эксперименте был проведён подбор гиперпараметров модели **CatBoostRegressor** с использованием библиотеки **Optuna**.

* Количество итераций: **500 trials**
* Метрика: **RMSE на логарифмированном таргете**
* Кросс-валидация: **5 фолдов**
* Целевая переменная:
  [
  y = \log(1 + SalePrice)
  ]

---

## 🏆 Лучшие параметры

```python
{
    'iterations': 920,
    'learning_rate': 0.06152055348604786,
    'depth': 4,
    'l2_leaf_reg': 7,
    'random_strength': 3,
    'subsample': 0.8
}
```

### Лучшее значение метрики

```
RMSE = 0.12100098503803247
```

---

## Анализ результатов

### 1. Стабильность решений

Топ-5 моделей показывают очень близкие значения:

| rank | RMSE     |
| ---- | -------- |
| 1    | 0.121001 |
| 2    | 0.121124 |
| 3    | 0.121125 |
| 4    | 0.121135 |
| 5    | 0.121183 |

👉 Разница между лучшей и 5-й моделью:
[
\approx 0.00018
]

Это означает:

* пространство гиперпараметров уже хорошо исследовано
* модель находится близко к локальному оптимуму
* дальнейший тюнинг даст очень небольшой прирост

---

### 2. Поведение гиперпараметров

#### 📌 depth = 4

* неглубокие деревья
* хорошая обобщающая способность
* снижает переобучение

#### 📌 learning_rate ≈ 0.06

* умеренное значение
* баланс между скоростью обучения и стабильностью

#### 📌 iterations ≈ 900–1000

* модель обучается достаточно долго
* но без сильного переобучения

#### 📌 subsample = 0.8

* используется стохастичность (bootstrap)
* это помогает обобщению

---

### 3. best_iteration_list

Пример:

```
[782, 847, 159, 732, 669]
```

Это означает:

* на каждом фолде оптимальное число деревьев разное
* модель не переобучается одинаково на всех данных
* есть вариативность → это нормально

---

## Важное наблюдение

Несмотря на 500 запусков Optuna:

улучшение после топ-5 моделей минимально

Это говорит о том, что:

* гиперпараметры уже не являются главным ограничением
* дальнейший рост качества требует:

  * ансамблей (bagging / stacking)
  * новых признаков (feature engineering)
  * других моделей (LightGBM, Ridge)


## Вывод

* CatBoost показывает стабильный результат ~**0.121**
* пространство гиперпараметров хорошо исследовано
* дальнейший рост качества требует **ансамблей или новых фичей**

---
